In [2]:
import xarray as xr, pandas as pd, numpy as np
from scipy.special import erf

# reload monthly temp + geopotential (already on disk)
ds_t = xr.open_dataset(r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\ERA5\extracted\data_stream-moda_stepType-avgua.nc")
t2m_c = ds_t["t2m"] - 273.15
tdim = "valid_time" if "valid_time" in t2m_c.dims else "time"
zg = xr.open_dataset("../Data/Raw/ERA5/era5_geopotential_alps.nc")
z_era5_m = zg["z"].squeeze() / 9.80665

df = pd.read_csv("C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Processed\stage2_model_ready.csv")
lon = xr.DataArray(df["CenLon"].values, dims="g")
lat = xr.DataArray(df["CenLat"].values, dims="g")
zmed = xr.DataArray(df["Zmed"].values, dims="g")

t_at = t2m_c.sel(longitude=lon, latitude=lat, method="nearest")
z_at = z_era5_m.sel(longitude=lon, latitude=lat, method="nearest")
t_corr = t_at - 0.0065 * (zmed - z_at)

sigma = 4.0
days = t_corr[tdim].dt.days_in_month
pdd = days * (sigma/np.sqrt(2*np.pi)*np.exp(-t_corr**2/(2*sigma**2))
              + t_corr/2*(1 + erf(t_corr/(sigma*np.sqrt(2)))))
apdd_yr = pdd.groupby(f"{tdim}.year").sum(tdim)

baseline = apdd_yr.sel(year=slice(2000, 2019)).mean("year")
df["apdd_2022"]     = apdd_yr.sel(year=2022).values
df["apdd_anom_2022"] = (apdd_yr.sel(year=2022) - baseline).values

pct = 100 * df["apdd_anom_2022"].mean() / baseline.mean().item()
print(df["apdd_anom_2022"].describe())
print(f"\nMean 2022 melt-energy increase vs 2000-2019: "
      f"+{df['apdd_anom_2022'].mean():.0f} °C·day (+{pct:.0f}%)")

df.to_csv("../Data/Processed/stage3_with_APDD.csv", index=False)
print("Saved with 2022 anomaly.")

count    3927.000000
mean      219.711779
std        54.759442
min        34.645302
25%       182.075029
50%       212.421899
75%       253.522601
max       617.328709
Name: apdd_anom_2022, dtype: float64

Mean 2022 melt-energy increase vs 2000-2019: +220 °C·day (+36%)
Saved with 2022 anomaly.


In [3]:
print(df[["apdd_anom_2022", "Zmed", "Area", "Slope"]].corr()["apdd_anom_2022"])

apdd_anom_2022    1.000000
Zmed             -0.491377
Area             -0.014807
Slope             0.044867
Name: apdd_anom_2022, dtype: float64


In [1]:
df["apdd_baseline"] = df["apdd"]          # the 2000-2019 mean you already computed
df["apdd_pct_change"] = 100 * df["apdd_anom_2022"] / df["apdd_baseline"]

print(df["apdd_pct_change"].describe())
print(df[["apdd_pct_change","Zmed"]].corr().iloc[0,1], "= correlation with elevation")

NameError: name 'df' is not defined

In [2]:
import os

wgms = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\WGMS\DOI-WGMS-FoG-2026-02-10 (1)"

print("Exists:", os.path.exists(wgms))
for root, dirs, files in os.walk(wgms):
    for f in files:
        print(os.path.join(root, f).replace(wgms, ""))

Exists: True
\datapackage.json
\readme.pdf
\data\agency.csv
\data\change.csv
\data\change_band.csv
\data\event.csv
\data\front_variation.csv
\data\glacier.csv
\data\mass_balance.csv
\data\mass_balance_band.csv
\data\mass_balance_point.csv
\data\person.csv
\data\state.csv
\data\state_band.csv


In [ ]:
import pandas as pd

WGMS = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\WGMS\DOI-WGMS-FoG-2026-02-10 (1)\data"

mb = pd.read_csv(WGMS + r"\mass_balance.csv")
gl = pd.read_csv(WGMS + r"\glacier.csv")

# link mass balance to RGI ids, filter to Alpine 2022
m = mb.merge(gl[["id", "rgi60_ids"]], left_on="glacier_id", right_on="id", how="left")
alpine_2022 = m[(m["rgi60_ids"].astype(str).str.startswith("RGI60-11")) &
                (m["year"] == 2022)]

print(len(alpine_2022), "Alpine glaciers with 2022 field measurements")
print(alpine_2022["annual_balance"].describe())

alpine_2022.to_csv("../Data/Processed/wgms_alpine_2022.csv", index=False)

In [1]:
import pandas as pd, os

path = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\WGMS\wgms-amce-2025-02b\wgms-amce-2025-02b\individual-glacier"   # point at the file you downloaded

print("Size (MB):", round(os.path.getsize(path)/1e6, 1))

d = pd.read_csv(path, nrows=5)      # peek first, in case it's large
print(d.columns.tolist())
print(d.head().to_string())

Size (MB): 0.1


PermissionError: [Errno 13] Permission denied: 'C:\\DATA\\Dissertation\\Glacier_Mass\\Glacier_Mass_Balance\\Glacier_Mass_Balance\\Data\\Raw\\WGMS\\wgms-amce-2025-02b\\wgms-amce-2025-02b\\individual-glacier'

In [11]:
import os

folder = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\WGMS\wgms-amce-2025-02b\wgms-amce-2025-02b\individual-glacier"

items = os.listdir(folder)
print("Number of items:", len(items))
print(items[:20])

Number of items: 121
['ACN_ArcticCanadaNorth_metadata.csv', 'ACN_gla_MEAN-CAL-mass-change-series_obs_unobs.csv', 'ACN_gla_mean-cal-mass-change_ANOM-ERROR_obs_unobs.csv', 'ACN_gla_mean-cal-mass-change_DH-ERROR_obs_unobs.csv', 'ACN_gla_mean-cal-mass-change_RHO-ERROR_obs_unobs.csv', 'ACN_gla_mean-cal-mass-change_TOTAL-ERROR_obs_unobs.csv', 'ACS_ArcticCanadaSouth_metadata.csv', 'ACS_gla_MEAN-CAL-mass-change-series_obs_unobs.csv', 'ACS_gla_mean-cal-mass-change_ANOM-ERROR_obs_unobs.csv', 'ACS_gla_mean-cal-mass-change_DH-ERROR_obs_unobs.csv', 'ACS_gla_mean-cal-mass-change_RHO-ERROR_obs_unobs.csv', 'ACS_gla_mean-cal-mass-change_TOTAL-ERROR_obs_unobs.csv', 'ALA_Alaska_metadata.csv', 'ALA_gla_MEAN-CAL-mass-change-series_obs_unobs.csv', 'ALA_gla_mean-cal-mass-change_ANOM-ERROR_obs_unobs.csv', 'ALA_gla_mean-cal-mass-change_DH-ERROR_obs_unobs.csv', 'ALA_gla_mean-cal-mass-change_RHO-ERROR_obs_unobs.csv', 'ALA_gla_mean-cal-mass-change_TOTAL-ERROR_obs_unobs.csv', 'ANT_AntarcticSubantarctic_metadata.cs

In [13]:
import os
folder = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\WGMS\wgms-amce-2025-02b\wgms-amce-2025-02b\individual-glacier"

ceu = [f for f in os.listdir(folder) if f.startswith("CEU")]
for f in ceu:
    print(f)

CEU_CentralEurope_metadata.csv
CEU_gla_MEAN-CAL-mass-change-series_obs_unobs.csv
CEU_gla_mean-cal-mass-change_ANOM-ERROR_obs_unobs.csv
CEU_gla_mean-cal-mass-change_DH-ERROR_obs_unobs.csv
CEU_gla_mean-cal-mass-change_RHO-ERROR_obs_unobs.csv
CEU_gla_mean-cal-mass-change_TOTAL-ERROR_obs_unobs.csv


In [14]:
import os, pandas as pd

folder = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\WGMS\wgms-amce-2025-02b\wgms-amce-2025-02b\individual-glacier"

f = os.path.join(folder, "CEU_gla_MEAN-CAL-mass-change-series_obs_unobs.csv")
d = pd.read_csv(f)

print("Shape:", d.shape)
print("Columns:", d.columns.tolist()[:25])
print()
print(d.iloc[:5, :12].to_string())

Shape: (3927, 116)
Columns: ['RGIId', 'REGION', 'CenLon', 'CenLat', 'Area', 'WGMS_ID', '1915', '1916', '1917', '1918', '1919', '1920', '1921', '1922', '1923', '1924', '1925', '1926', '1927', '1928', '1929', '1930', '1931', '1932', '1933']

            RGIId REGION   CenLon   CenLat   Area   WGMS_ID      1915      1916      1917      1918      1919      1920
0  RGI60-11.00001    CEU  13.5987  47.4949  0.122  134185.0  0.697537  0.846537 -0.251463  0.490537  0.128747 -0.083214
1  RGI60-11.00002    CEU  13.6135  47.4845  2.292     535.0  1.059062  1.208062  0.110062  0.852062  0.490330  0.278286
2  RGI60-11.00003    CEU  13.5960  47.4835  0.851  134186.0  0.788601  0.937601 -0.160399  0.581601  0.219742  0.007880
3  RGI60-11.00004    CEU  13.5829  47.4807  0.053  134187.0  1.207194  1.356194  0.258194  1.000194  0.638243  0.426513
4  RGI60-11.00005    CEU  13.6026  47.4774  0.057  134188.0  1.159524  1.308524  0.210524  0.952524  0.590677  0.378799
